# Metaheuristics for Multi-Resource Workload Allocation

## Problem Definition

This coursework investigates a multi-resource model selection problem.
Each candidate model has a utility value and consumes several resources.
The objective is to select a subset of models that maximises total utility
without exceeding any resource capacity.

A solution is represented by a binary vector: 1 means a model is selected,
and 0 means it is not selected.

## 1. Data loading

NumPy is used to store and calculate model utilities and resource demands.
The loading function follows the structure of the lecturer-provided loader,
with an additional check on the number of data entries.

The file contains the number of models (n), the number of resources (m),
n utility values, m × n resource demands, and m capacity limits, in that order.
The resource demand matrix has shape (m, n): each row represents a resource,
and each column represents a model.

The two lecturer-provided data files are included alongside this
notebook. Keep both files in the same directory as the notebook
and run the notebook from that directory.

In [2]:
import numpy as np


def load_workload_instance(file_path):
    """Read model utilities, resource demands and capacities."""
    with open(file_path, "r", encoding="utf-8") as f:
        tokens = f.read().split()

    n = int(tokens[0])
    m = int(tokens[1])

    expected_count = 2 + n + m * n + m
    if len(tokens) != expected_count:
        raise ValueError("The data quantity is incorrect. Please check the data file.")

    idx = 2

    utilities = np.array(
        tokens[idx:idx + n], dtype=float
    )
    idx += n

    demands = np.array(
        tokens[idx:idx + m * n], dtype=float
    ).reshape(m, n)
    idx += m * n

    capacities = np.array(
        tokens[idx:idx + m], dtype=float
    )

    return n, m, utilities, demands, capacities

## 2. Instance 1: Data Summary

Instance 1 contains 28 candidate models and two resource constraints,
each with a capacity of 500. It is used to develop and evaluate the
baseline hill-climbing algorithm in Part 1.

The following output reports the instance dimensions and capacity limits
to check that they match the assignment specification.

In [3]:
n, m, utilities, demands, capacities = load_workload_instance(
   "multi_resource_n28_m2.txt"
)

print("Number of models:", n)
print("Number of resources:", m)
print("Resource capacities:", capacities)

Number of models: 28
Number of resources: 2
Resource capacities: [500. 500.]


## 3. Solution Representation and Evaluation

A solution is represented by a binary vector of length n.
A value of 1 selects a model, while 0 excludes it.

The total utility and resource usage are calculated by summing
the contributions of the selected models. A solution is feasible
only if every resource usage is within its capacity.

Selecting models 1 and 3 gives a total utility of 24,405 and resource usage of 130 and 155. Both resource usages are below their respective capacities of 500, so the solution is feasible.

This manually constructed example checks the evaluation calculation. It is not a claim that the solution is the best.

In [4]:
def evaluate_solution(x, utilities, demands, capacities):
    """Return total utility, resource usage and feasibility."""
    total_utility = utilities @ x
    resource_usage = demands @ x
    feasible = bool(np.all(resource_usage <= capacities))

    return total_utility, resource_usage, feasible


# Start with all models unselected.
x = np.zeros(n, dtype=int)

# Select model 1 and model 3.
x[0] = 1
x[2] = 1

value, usage, feasible = evaluate_solution(
    x, utilities, demands, capacities
)

print("Selection vector:", x)
print("Total utility:", value)
print("Resource usage:", usage)
print("Feasible:", feasible)

Selection vector: [1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Total utility: 24405.0
Resource usage: [130. 155.]
Feasible: True


### Feasibility Check

To check the constraint evaluation, all 28 models are selected.
This deliberately creates an infeasible solution, allowing us
to verify that the evaluation function detects resource violations.

In [5]:
# Select all 28 models.
x_all = np.ones(n, dtype=int)

value_all, usage_all, feasible_all = evaluate_solution(
    x_all, utilities, demands, capacities
)

print("Resource usage:", usage_all)
print("Resource capacities:", capacities)
print("Feasible:", feasible_all)

Resource usage: [1125.  995.]
Resource capacities: [500. 500.]
Feasible: False


Selecting all 28 models uses 1,125 units of resource 1 and 995 units of resource 2. Both exceed their capacities of 500, so the evaluation function correctly identifies this solution as infeasible.

## Part 1: Baseline Discrete Local Search

This part implements and evaluates a first-improvement hill climber
on Instance 1. It covers feasible initialisation, neighbourhood
operators, constraint handling, and performance across 30 independent runs.

### 4. Randomised Feasible Initialisation

The initial solution is constructed from an empty selection.
Models are considered in a random order and added only when all
resource constraints remain satisfied.

This provides a feasible starting point. Different random seeds
can produce different initial solutions. The empty solution and
each attempted addition are included in the evaluation count.

In [6]:
def create_initial_solution(utilities, demands, capacities, rng):
    n = len(utilities)
    x = np.zeros(n, dtype=int)

    value, usage, _ = evaluate_solution(
        x, utilities, demands, capacities
    )
    evaluations = 1

    for j in rng.permutation(n):
        candidate = x.copy()
        candidate[j] = 1

        new_value, new_usage, feasible = evaluate_solution(
            candidate, utilities, demands, capacities
        )
        evaluations += 1

        if feasible:
            x, value, usage = candidate, new_value, new_usage

    return x, value, usage, evaluations

### 5. Neighbourhood Operators

Two neighbourhood operators are used:

- Single-bit flip: change one model's selection status.
- One-for-one swap: remove one selected model and add one unselected model.

A swap can improve a capacity-limited solution when adding a model
alone would violate a constraint. Neighbours are evaluated during
the search, and infeasible candidates are rejected.

In [7]:
def generate_neighbors(x, rng):
    # Operator 1: flip one selection decision.
    for j in rng.permutation(len(x)):
        candidate = x.copy()
        candidate[j] = 1 - candidate[j]
        yield candidate

    # Operator 2: exchange one selected and one unselected model.
    selected = rng.permutation(np.flatnonzero(x == 1))
    unselected = rng.permutation(np.flatnonzero(x == 0))

    for remove in selected:
        for add in unselected:
            candidate = x.copy()
            candidate[remove] = 0
            candidate[add] = 1
            yield candidate

### 6. First-Improvement Hill Climbing

The search starts from a randomly constructed feasible solution.
Neighbours are examined in a randomised order: single-bit flips
are considered first, followed by one-for-one swaps.

The first feasible neighbour with strictly higher utility is accepted,
and neighbourhood exploration restarts from that solution.
Infeasible and non-improving candidates are rejected.

The search stops when a complete neighbourhood contains no improving
feasible solution, or when the evaluation budget is exhausted.
Initialisation and all evaluated neighbours count towards this budget.

Strict improvement prevents cycling, but the search may stop at a local optimum rather than a global optimum (Russell & Norvig, 2021).

In [8]:
def hill_climb(utilities, demands, capacities, seed=42,
               max_evaluations=5000):

    if max_evaluations < len(utilities) + 1:
        raise ValueError("Budget is too small for initialisation.")

    rng = np.random.default_rng(seed)

    x, value, usage, evaluations = create_initial_solution(
        utilities, demands, capacities, rng
    )

    initial_value = value
    history = [(evaluations, value)]
    stop_reason = "evaluation_budget"

    while evaluations < max_evaluations:
        improved = False

        for candidate in generate_neighbors(x, rng):
            if evaluations >= max_evaluations:
                break

            new_value, new_usage, feasible = evaluate_solution(
                candidate, utilities, demands, capacities
            )
            evaluations += 1

            if feasible and new_value > value:
                x, value, usage = candidate, new_value, new_usage
                improved = True

            history.append((evaluations, value))

            if improved:
                break

        if not improved:
            if evaluations < max_evaluations:
                stop_reason = "local_optimum"
            break

    return {
        "solution": x,
        "initial_value": initial_value,
        "value": value,
        "usage": usage,
        "evaluations": evaluations,
        "history": history,
        "stop_reason": stop_reason,
    }

### 7. Single-Run Demonstration

The hill climber is applied to Instance 1 using seed 42 and
a maximum budget of 5,000 evaluations.

This demonstration reports the improvement from the initial solution,
the final resource usage, and the reason for termination.
Multiple independent runs will be used later to assess performance.

In [10]:
result = hill_climb(
    utilities,
    demands,
    capacities,
    seed=42,
    max_evaluations=5000
)

print("Initial utility:", result["initial_value"])
print("Final utility:", result["value"])
print("Resource usage:", result["usage"])
print("Resource capacities:", capacities)
print("Evaluations:", result["evaluations"])
print("Stop reason:", result["stop_reason"])

print(
    "Selected models:",
    np.flatnonzero(result["solution"]) + 1
)

# Independently check the returned solution.
checked_value, checked_usage, checked_feasible = evaluate_solution(
    result["solution"], utilities, demands, capacities
)

print("Final solution feasible:", checked_feasible)

Initial utility: 74163.0
Final utility: 98003.0
Resource usage: [455. 494.]
Resource capacities: [500. 500.]
Evaluations: 498
Stop reason: local_optimum
Selected models: [ 1  5  6  7  8 10 11 12 13 14 15 16 17 18 19 20 23 24 27]
Final solution feasible: True


For seed 42, hill climbing increased utility from 74,163 to 98,003.
The final resource usages were 455 and 494, satisfying both
capacity limits of 500.

The search terminated after 498 evaluations because no improving
feasible neighbour remained under the defined neighbourhood.
This establishes local optimality for these operators, not global
optimality. A single run is insufficient to assess reliability.

### 8. Repeated Experiments

The hill climber is evaluated on Instance 1 over 30 independent runs,
using random seeds 0–29. Each run starts from a newly constructed
feasible solution and has a maximum budget of 5,000 evaluations,
including initialisation.

Runs terminate early when no improving feasible neighbour remains.
Final utility, evaluation count, termination reason and elapsed
search time are recorded. Final solutions are checked for feasibility.

Post-run validation is excluded from search time and evaluation counts.

In [11]:
from time import perf_counter

seeds = list(range(30))
budget = 5000
runs = []

for seed in seeds:
    start = perf_counter()

    run = hill_climb(
        utilities,
        demands,
        capacities,
        seed=seed,
        max_evaluations=budget
    )

    elapsed = perf_counter() - start

    # Check the returned solution after timing the search.
    checked_value, checked_usage, checked_feasible = evaluate_solution(
        run["solution"], utilities, demands, capacities
    )

    if not checked_feasible or not np.isclose(checked_value, run["value"]):
        raise RuntimeError(f"Invalid result for seed {seed}")

    run["seed"] = seed
    run["seconds"] = elapsed
    run["feasible"] = checked_feasible
    runs.append(run)

print(f"Completed {len(runs)} runs.")
print(f"Maximum evaluation budget per run: {budget}")

Completed 30 runs.
Maximum evaluation budget per run: 5000


### 9. Performance Summary

Final utility is summarised using the best, worst, mean, median
and sample standard deviation across runs. The standard deviation
describes variability between runs.

The mean improvement over initialisation, evaluation count and
elapsed search time are also reported. The best observed solution
is inspected for resource usage.

In [12]:
initial_values = np.array([run["initial_value"] for run in runs])
final_values = np.array([run["value"] for run in runs])
evaluation_counts = np.array([run["evaluations"] for run in runs])
times = np.array([run["seconds"] for run in runs])

gains = final_values - initial_values

# Show the results of individual runs.
print(
    f"{'Seed':>4} {'Initial':>10} {'Final':>10} "
    f"{'Gain':>10} {'Evaluations':>12} {'Seconds':>10}"
)

for run in runs:
    print(
        f"{run['seed']:4d} "
        f"{run['initial_value']:10.0f} "
        f"{run['value']:10.0f} "
        f"{run['value'] - run['initial_value']:10.0f} "
        f"{run['evaluations']:12d} "
        f"{run['seconds']:10.4f}"
    )

print("\nSummary across 30 runs")
print(f"Best final utility: {final_values.max():.0f}")
print(f"Worst final utility: {final_values.min():.0f}")
print(f"Mean final utility: {final_values.mean():.2f}")
print(f"Median final utility: {np.median(final_values):.2f}")
print(f"Sample standard deviation: {final_values.std(ddof=1):.2f}")
print(f"Mean improvement: {gains.mean():.2f}")
print(f"Mean evaluations: {evaluation_counts.mean():.2f}")
print(f"Mean search time (seconds): {times.mean():.4f}")

print(
    "Feasible final solutions:",
    sum(run["feasible"] for run in runs),
    "/",
    len(runs)
)

print(
    "Local-optimum stops:",
    sum(run["stop_reason"] == "local_optimum" for run in runs)
)

print(
    "Budget stops:",
    sum(run["stop_reason"] == "evaluation_budget" for run in runs)
)

# Inspect the best result among these 30 runs.
best_run = max(runs, key=lambda run: run["value"])

print("\nBest observed run")
print("Seed:", best_run["seed"])
print("Selected models:", np.flatnonzero(best_run["solution"]) + 1)
print("Resource usage:", best_run["usage"])

print(
    "Resource utilisation (%):",
    np.round(100 * best_run["usage"] / capacities, 1)
)

Seed    Initial      Final       Gain  Evaluations    Seconds
   0      60460      91232      30772          698     0.0103
   1      77945     109817      31872          762     0.0101
   2      99822     121164      21342          919     0.0123
   3      96849     110154      13305          612     0.0086
   4      93727     102737       9010          760     0.1041
   5     102492     115971      13479          716     0.0174
   6      77249     117360      40111         1099     0.0211
   7      95108      95108          0          217     0.0041
   8      73037      79767       6730          560     0.0098
   9      98259     110032      11773          491     0.0090
  10      99513     109152       9639          573     0.0100
  11      61363     101609      40246          968     0.0178
  12      85817     113286      27469          627     0.0125
  13     107994     113584       5590          602     0.0115
  14     105577     120844      15267          604     0.0120
  15    